In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import(
    trim,
    col,
    row_number
)
from pyspark.sql.types import StringType

##### Importing from utils and validation_utils package

In [0]:
from utils.custom_utils import Transformations
from validation_utils.test_utils import Validations
tr_obj = Transformations()
va_obj = Validations()

#### Reading bronze table

In [0]:
df = spark.table("lakehouse.bronze.cust_info")

### Silver Layer Transformations

#### 1. Renaming Columns

In [0]:
rename_config = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

for old_name, new_name in rename_config.items():
    df = df.withColumnRenamed(old_name, new_name) 

#### 2. Trimming

- Check if trimming is required using whitespace_check function

In [0]:
df = tr_obj.trim_columns(df)

#### 3. Removing records with no customer_id

- Check if removing nulls is required using null_check function

In [0]:
df = tr_obj.remove_null(df = df, primary_col = "customer_id")

#### 4. Removing customer_id duplicates
- Since create_date behaves as last_updated in our data source, hence it can be used as CDC(Change Data Capture)

- Check if there are duplicate values of customer_id using duplicate_check function

In [0]:
df = tr_obj.dedup(df = df, dedup_cols = ["customer_id"], cdc = "create_date")

#### 5. Normalization

In [0]:
df = (
    df.withColumn(
        "marital_status",
        when(upper(col("marital_status")) == "M", "Married")
        .when(upper(col("marital_status")) == "S", "Single")
        .otherwise("n/a")
    )
    .withColumn(
        "gender",
        when(upper(col("gender")) == "M", "Male")
        .when(upper(col("gender")) == "F", "Female")
        .otherwise("n/a")
    )
)

#### 6. Adding ingestion_ts column

In [0]:
df = tr_obj.add_ingestion_timestamp(df)

#### DataFrame Sanity Check

In [0]:
df.limit(10).display()

#### 7. Applying upsert(SCD type 1)

In [0]:
target_table = "lakehouse.silver.crm_customers"

if spark.catalog.tableExists(target_table):
    tr_obj.upsert(
        spark = spark,
        df = df,
        key_cols = ["customer_id"],
        table = "crm_cutomers",
        cdc = "create_date"
    )
else:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

#### Silver table sanity check

In [0]:
%sql
select *
from lakehouse.silver.crm_customers
limit 10